# 01 - Explore the raw data
What each year contains, who was asked the AI questions (skip logic), and the 2023 column swap.
Raw files: `data/raw/<year>/` (download with `python -m src.download`).

In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from src.load import load_year, RAW_COLS
from src.config import RAW
for y in (2023, 2024, 2025):
    schema = pd.read_csv(RAW / str(y) / "schema.csv", dtype=str)
    ai = schema[schema.qname.str.startswith("AI", na=False)][["qname", "question"]].drop_duplicates("qname")
    display(Markdown(f"### {y}: AI questions in the schema")); display(ai.head(12))

### 2023: AI questions in the schema

,qname,question
36,AISearch,Which <b>AI-powered search tools</b> did you use regularly over the past yea...
37,AIDev,Which <b>AI-powered developer tools</b> did you use regularly over the past ...
46,AISelect,Do you currently use AI tools in your development process? *
47,AISent,How favorable is your stance on using AI tools as part of your development w...
48,AIAcc,"For the AI tools you use as part of your development workflow, what are the ..."
49,AIBen,How much do you trust the accuracy of the output from AI tools as part of yo...
50,AITool,Which parts of your development workflow are you currently using AI tools fo...
51,AINext,"Thinking about how your workflow and process changes over time, how similar ..."
52,AIOpen,"Please describe how you would expect your workflow to be different, if at al..."


### 2024: AI questions in the schema

,qname,question
32,AISearchDev,Which <b>AI-powered search and developer tools</b> did you use regularly ove...
39,AISelect,Do you currently use AI tools in your development process? *
40,AISent,How favorable is your stance on using AI tools as part of your development w...
41,AIBen,"For the AI tools you use as part of your development workflow, what are the ..."
42,AIAcc,How much do you trust the accuracy of the output from AI tools as part of yo...
43,AIComplex,How well do the AI tools you use in your development workflow handle complex...
44,AITool,Which parts of your development workflow are you currently using AI tools fo...
45,AINext,"Thinking about how your job and process changes over time, how integrated in..."
46,AIThreat,Do you believe AI is a threat to your current job?
47,AIEthics,Which AI ethical responsibilities are most important to you? Select all tha...


### 2025: AI questions in the schema

,qname,question
58,AILearnHow,How did you learn to code for AI in the past year? Select all that apply.
68,AIThreat,Do you believe AI is a threat to your current job?
105,AIModelsChoice,Did you utilize or integrate specific Large Language Models (LLMs) to power ...
106,AIModels,Which <b>LLM models for AI tools</b> have you used for development work in t...
107,AIModelsHaveEntry,"Was the LLM model for AI tools you use not listed above? List below, separa..."
108,AIModelsWantEntry,Was the LLM model for AI tools you want to use not listed above? List below...
116,AISelect,Do you currently use AI tools in your development process?
117,AISent,How favorable is your stance on using AI tools as part of your development w...
118,AIAcc,How much do you trust the accuracy of the output from AI tools as part of yo...
119,AIComplex,How well do the AI tools you use in your development workflow handle complex...


## The 2023 column swap
In 2023 the *trust* answers are stored in `AIBen` and the *benefits* answers in `AIAcc` - the reverse of 2024/2025.

In [3]:
raw23 = load_year(2023)
display(raw23["AIBen"].value_counts().head(6).to_frame("2023 AIBen (actually TRUST)"))
display(raw23["AIAcc"].str.split(";").explode().value_counts().head(6).to_frame("2023 AIAcc (actually BENEFITS)"))

,2023 AIBen (actually TRUST)
AIBen,
Somewhat trust,24128
Neither trust nor distrust,18837
Somewhat distrust,13330
Highly distrust,3350
Highly trust,1751


,2023 AIAcc (actually BENEFITS)
AIAcc,
Increase productivity,32509
Speed up learning,24938
Greater efficiency,24739
Improve accuracy in coding,13189
Improve collaboration,3721
Other (please explain),2131


## Skip logic: who answered the trust question?
This decides the denominator of every trust percentage.

In [4]:
trust_col = {2023: "AIBen", 2024: "AIAcc", 2025: "AIAcc"}
for y in (2023, 2024, 2025):
    r = load_year(y)
    t = pd.crosstab(r["AISelect"].fillna("(blank)"), r[trust_col[y]].notna().map({True: "answered trust", False: "no trust answer"}))
    display(Markdown(f"**{y}**")); display(t)

**2023**

AIBen,answered trust,no trust answer
AISelect,,
(blank),0,1211
"No, and I don't plan to",0,26221
"No, but I plan to soon",22508,202
Yes,38888,154


**2024**

AIAcc,answered trust,no trust answer
AISelect,,
(blank),0,4530
"No, and I don't plan to",0,14837
"No, but I plan to soon",0,8408
Yes,37302,360


**2025**

AIAcc,answered trust,no trust answer
AISelect,,
(blank),37,15434
"No, and I don't plan to",5380,74
"No, but I plan to soon",1754,43
"Yes, I use AI tools daily",15662,221
"Yes, I use AI tools monthly or infrequently",4563,65
"Yes, I use AI tools weekly",5901,57


**Conclusion:** 2023 = users + planners, 2024 = users only, 2025 = everyone who answered AISelect. The only population asked in all three years is **current AI users**, so trust trends use that base.

## Benchmark: do our harmonised numbers match the official tables?

In [5]:
bench = pd.read_csv("reports/tables/benchmark_check.csv")
print(bench.status.value_counts().to_dict(), "max |diff| =", bench.diff_pp.abs().max(), "pp")
bench[bench.group == "All Respondents"].head(30)

{'PASS': 84} max |diff| = 0.6 pp


,year,question,group,answer,official_pct,our_pct,official_base,our_base,diff_pp,status
0,2023,AISelect,All Respondents,Yes,43.78,44.38,89184,87973,0.60,PASS
1,2023,AISelect,All Respondents,Plan to,25.46,25.81,89184,87973,0.35,PASS
2,2023,AISelect,All Respondents,No,29.40,29.81,89184,87973,0.41,PASS
6,2023,AISent,All Respondents,5 Very favorable,27.72,27.72,61501,61501,0.00,PASS
7,2023,AISent,All Respondents,4 Favorable,48.56,48.56,61501,61501,0.00,PASS
8,2023,AISent,All Respondents,3 Indifferent,16.50,16.50,61501,61501,0.00,PASS
9,2023,AISent,All Respondents,Unsure,4.02,4.02,61501,61501,0.00,PASS
10,2023,AISent,All Respondents,2 Unfavorable,2.76,2.76,61501,61501,0.00,PASS
11,2023,AISent,All Respondents,1 Very unfavorable,0.44,0.44,61501,61501,0.00,PASS
18,2023,AIBen,All Respondents,5 Highly trust,2.85,2.85,39042,61396,0.00,PASS
